<a href="https://colab.research.google.com/github/Wasayyyyyy/CV-Tasks/blob/main/Task_1_FA23_BAI_043.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skin Lesion Classification — Transfer Learning Benchmark
This notebook reproduces the three comparison tables from `Task_01.docx`:
- **Table 1** — Transfer learning CNN backbones (AlexNet, VGG16, VGG19, ResNet18/50/101, DenseNet121, EfficientNet-B0)
- **Table 2** — Deep-feature extraction + classical ML classifiers
- **Table 3** — Computational efficiency (params, size, FLOPs, inference time)

Dataset: `nodoubttome/skin-cancer9-classesisic` (Kaggle, 9-class ISIC skin cancer dataset).

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. Then run
`Runtime → Run all` and check the very first printed line of the setup cell
says `Using device: cuda` — if it says `cpu`, the whole notebook will be
10-50x slower and you should fix the runtime before doing anything else.

### Speed notes (read this if training feels slow)
- By default this notebook uses the standard **frozen-backbone transfer
  learning** approach: the pretrained CNN body is frozen and only the new
  classification head is trained. This is what "transfer learning" means in
  the comparison-table sense, it's the standard baseline setup, and it's
  roughly 10-20x faster than fine-tuning every layer. Set
  `FREEZE_BACKBONE = False` in the training cell if you specifically want
  full fine-tuning (much slower, marginal accuracy gain here).
- `EPOCHS = 8` with early stopping (patience 3) is enough for a frozen head
  to converge — increase it later if you want to squeeze out more accuracy.
- Mixed precision (`torch.cuda.amp`) is used automatically on GPU.
- Table 2's SVM classifiers are the classic bottleneck of this kind of
  notebook — `SVC` scales poorly with sample count. This version subsamples
  the training features for the two SVMs and uses `LinearSVC` for the linear
  case, which is dramatically faster with a small accuracy trade-off.


In [13]:
# ============================================================
# 1. Install dependencies
# ============================================================
!pip install -q kagglehub thop xgboost scikit-learn --upgrade


In [2]:
# ============================================================
# 2. Imports
# ============================================================
import os, time, copy, json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sklearn.preprocessing import label_binarize

from thop import profile

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
# ============================================================
# 3. Download dataset
# ============================================================
import kagglehub

path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

# Inspect the folder structure so we can point ImageFolder at the right dirs
for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)" if files else "")


Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
skin-cancer9-classesisic/ 
  Skin cancer ISIC The International Skin Imaging Collaboration/ 
    Test/ 
    Train/ 


In [4]:
# ============================================================
# 4. Locate train/test directories
# ============================================================
# The Kaggle "Skin Cancer ISIC" dataset ships as:
#   <path>/Skin cancer ISIC The International Skin Imaging Collaboration/Train
#   <path>/Skin cancer ISIC The International Skin Imaging Collaboration/Test
# Adjust these two lines if kagglehub's folder naming differs on your run
# (the printout from the cell above shows the exact names).

def find_dir(base, keyword):
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            if keyword.lower() in d.lower():
                return os.path.join(root, d)
    raise FileNotFoundError(f"Could not find a folder containing '{keyword}' under {base}")

train_dir = find_dir(path, "train")
test_dir  = find_dir(path, "test")
print("Train dir:", train_dir)
print("Test dir :", test_dir)

CLASS_NAMES = sorted(os.listdir(train_dir))
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes:", CLASS_NAMES)


Train dir: /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train
Test dir : /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test
9 classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


In [5]:
# ============================================================
# 5. Transforms, datasets, train/val split
# ============================================================
IMG_SIZE = 224
BATCH_SIZE = 64   # bump to 128 if you have a bigger GPU (A100/L4) and hit no OOM

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

full_train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
full_train_eval_ds = datasets.ImageFolder(train_dir, transform=eval_tfms)  # same images, eval transform
test_ds  = datasets.ImageFolder(test_dir, transform=eval_tfms)

# stratified 90/10 split of the training folder into train/val
targets = np.array(full_train_ds.targets)
idx_all = np.arange(len(targets))
train_idx, val_idx = train_test_split(idx_all, test_size=0.1, stratify=targets, random_state=SEED)

train_subset = Subset(full_train_ds, train_idx)
val_subset   = Subset(full_train_eval_ds, val_idx)

NUM_WORKERS = 4
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,      batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

print(f"Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_ds)}")


Train: 2015 | Val: 224 | Test: 118


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [6]:
# ============================================================
# 6. Model builders (feature dim recorded for Table 2 later)
# ============================================================
def freeze_backbone(model, name):
    """Freezes every pretrained layer, leaving only the newly-added
    classification head trainable. This is the standard 'transfer learning
    as feature extraction' setup and is what makes training fast."""
    name = name.lower()
    for p in model.parameters():
        p.requires_grad = False

    if name in ("alexnet", "vgg16", "vgg19"):
        head = model.classifier[-1]
    elif name in ("resnet18", "resnet50", "resnet101"):
        head = model.fc
    elif name == "densenet121":
        head = model.classifier
    elif name == "efficientnet-b0":
        head = model.classifier[-1]
    else:
        raise ValueError(name)

    for p in head.parameters():
        p.requires_grad = True
    return model


def build_model(name, num_classes=NUM_CLASSES):
    """Returns (model, feature_dim) with the pretrained ImageNet backbone
    and a fresh classification head sized for our dataset."""
    name = name.lower()

    if name == "alexnet":
        m = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier[6].in_features
        m.classifier[6] = nn.Linear(feat_dim, num_classes)

    elif name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier[6].in_features
        m.classifier[6] = nn.Linear(feat_dim, num_classes)

    elif name == "vgg19":
        m = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier[6].in_features
        m.classifier[6] = nn.Linear(feat_dim, num_classes)

    elif name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        feat_dim = m.fc.in_features
        m.fc = nn.Linear(feat_dim, num_classes)

    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        feat_dim = m.fc.in_features
        m.fc = nn.Linear(feat_dim, num_classes)

    elif name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
        feat_dim = m.fc.in_features
        m.fc = nn.Linear(feat_dim, num_classes)

    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier.in_features
        m.classifier = nn.Linear(feat_dim, num_classes)

    elif name == "efficientnet-b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(feat_dim, num_classes)

    else:
        raise ValueError(f"Unknown model: {name}")

    return m, feat_dim

MODEL_NAMES_T1 = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50",
                   "ResNet101", "DenseNet121", "EfficientNet-B0"]
# Table 3 in the task doc omits ResNet101
MODEL_NAMES_T3 = [m for m in MODEL_NAMES_T1 if m != "ResNet101"]


In [7]:
# ============================================================
# 7. Train / evaluate helpers
# ============================================================
EPOCHS = 8           # frozen-backbone heads converge fast; raise if you fine-tune fully
LR = 1e-3            # higher LR is fine/needed since we're only training a small head
PATIENCE = 3         # early stopping patience on val loss
FREEZE_BACKBONE = True   # False = full fine-tuning (10-20x slower, marginal gain here)

USE_AMP = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_model(model, name, train_loader, val_loader, epochs=EPOCHS, lr=LR,
                 freeze=FREEZE_BACKBONE):
    if freeze:
        model = freeze_backbone(model, name)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
    else:
        trainable_params = model.parameters()

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(trainable_params, lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                       factor=0.1, patience=2)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        run_loss, run_correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * x.size(0)
            run_correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
        train_loss, train_acc = run_loss / n, run_correct / n

        model.eval()
        v_loss, v_correct, vn = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    out = model(x)
                    loss = criterion(out, y)
                v_loss += loss.item() * x.size(0)
                v_correct += (out.argmax(1) == y).sum().item()
                vn += x.size(0)
        val_loss, val_acc = v_loss / vn, v_correct / vn
        scheduler.step(val_loss)

        dt = time.time() - t0
        print(f"  epoch {epoch+1:02d}/{epochs} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
              f"| val_loss {val_loss:.4f} acc {val_acc:.4f} | {dt:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print("  early stopping")
                break

    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def get_probs_labels(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def compute_metrics(y_true, probs, num_classes=NUM_CLASSES):
    y_pred = probs.argmax(1)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    try:
        y_bin = label_binarize(y_true, classes=list(range(num_classes)))
        auc = roc_auc_score(y_bin, probs, average="weighted", multi_class="ovr")
    except Exception:
        auc = np.nan
    return {"Accuracy (%)": acc*100, "Precision (%)": prec*100,
            "Recall (%)": rec*100, "F1-Score (%)": f1*100, "AUC (%)": auc*100}


/tmp/ipykernel_519/1316555479.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [8]:
# ============================================================
# 8. TABLE 1 — train & evaluate all 8 backbones
# ============================================================
table1_rows = []
trained_models = {}   # keep the trained weights around for Table 2 feature extraction

for name in MODEL_NAMES_T1:
    t_start = time.time()
    print(f"\n=== Training {name} ===")
    model, _ = build_model(name)
    model = train_model(model, name, train_loader, val_loader)
    probs, labels = get_probs_labels(model, test_loader)
    metrics = compute_metrics(labels, probs)
    metrics["Model"] = name
    table1_rows.append(metrics)
    trained_models[name] = model.cpu()   # move off GPU to free memory between runs
    torch.cuda.empty_cache()
    print(f"  >>> {name} done in {(time.time()-t_start)/60:.1f} min")

table1 = pd.DataFrame(table1_rows)[["Model", "Accuracy (%)", "Precision (%)",
                                     "Recall (%)", "F1-Score (%)", "AUC (%)"]]
table1 = table1.round(2)
table1



=== Training AlexNet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:02<00:00, 99.2MB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_519/1316555479.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_519/1316555479.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  epoch 01/8 | train_loss 1.7088 acc 0.3970 | val_loss 1.4939 acc 0.4821 | 38.5s
  epoch 02/8 | train_loss 1.3440 acc 0.5216 | val_loss 1.3994 acc 0.5089 | 30.6s
  epoch 03/8 | train_loss 1.2918 acc 0.5459 | val_loss 1.3835 acc 0.5089 | 31.0s
  epoch 04/8 | train_loss 1.2344 acc 0.5663 | val_loss 1.4140 acc 0.5670 | 34.5s
  epoch 05/8 | train_loss 1.2241 acc 0.5702 | val_loss 1.4028 acc 0.5536 | 32.1s
  epoch 06/8 | train_loss 1.1877 acc 0.5712 | val_loss 1.4141 acc 0.5268 | 31.3s
  early stopping
  >>> AlexNet done in 3.5 min

=== Training VGG16 ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 96.5MB/s]
/tmp/ipykernel_519/1316555479.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_519/1316555479.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  epoch 01/8 | train_loss 1.8886 acc 0.3132 | val_loss 1.6980 acc 0.3750 | 32.9s
  epoch 02/8 | train_loss 1.6486 acc 0.4074 | val_loss 1.6566 acc 0.4464 | 33.9s
  epoch 03/8 | train_loss 1.5996 acc 0.4352 | val_loss 1.5217 acc 0.4420 | 32.3s
  epoch 04/8 | train_loss 1.5516 acc 0.4511 | val_loss 1.5253 acc 0.4062 | 33.6s
  epoch 05/8 | train_loss 1.5686 acc 0.4342 | val_loss 1.5084 acc 0.4375 | 32.9s
  epoch 06/8 | train_loss 1.4974 acc 0.4561 | val_loss 1.4905 acc 0.4911 | 34.2s
  epoch 07/8 | train_loss 1.5030 acc 0.4556 | val_loss 1.4638 acc 0.4955 | 33.1s
  epoch 08/8 | train_loss 1.4801 acc 0.4734 | val_loss 1.4452 acc 0.5045 | 34.1s
  >>> VGG16 done in 4.7 min

=== Training VGG19 ===
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 129MB/s]
/tmp/ipykernel_519/1316555479.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_519/1316555479.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  epoch 01/8 | train_loss 1.9543 acc 0.2730 | val_loss 1.7585 acc 0.3884 | 33.5s
  epoch 02/8 | train_loss 1.5974 acc 0.4625 | val_loss 1.5614 acc 0.4955 | 32.0s
  epoch 03/8 | train_loss 1.4351 acc 0.5275 | val_loss 1.4849 acc 0.5089 | 33.2s
  epoch 04/8 | train_loss 1.3207 acc 0.5573 | val_loss 1.4044 acc 0.5312 | 32.0s
  epoch 05/8 | train_loss 1.2692 acc 0.5717 | val_loss 1.3664 acc 0.5491 | 32.9s
  epoch 06/8 | train_loss 1.2452 acc 0.5826 | val_loss 1.3171 acc 0.5580 | 32.0s
  epoch 07/8 | train_loss 1.1697 acc 0.6149 | val_loss 1.3255 acc 0.5759 | 31.9s
  epoch 08/8 | train_loss 1.1400 acc 0.6149 | val_loss 1.2646 acc 0.6027 | 33.3s
  >>> DenseNet121 done in 4.5 min

=== Training EfficientNet-B0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]
/tmp/ipykernel_519/1316555479.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
/tmp/ipykernel_519/1316555479.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  epoch 01/8 | train_loss 1.8294 acc 0.3702 | val_loss 1.6374 acc 0.4732 | 44.0s
  epoch 02/8 | train_loss 1.4835 acc 0.5231 | val_loss 1.4816 acc 0.5312 | 31.5s
  epoch 03/8 | train_loss 1.3377 acc 0.5722 | val_loss 1.4138 acc 0.5536 | 31.6s
  epoch 04/8 | train_loss 1.2811 acc 0.5797 | val_loss 1.3633 acc 0.5446 | 32.7s
  epoch 05/8 | train_loss 1.2079 acc 0.6089 | val_loss 1.3343 acc 0.5580 | 31.4s
  epoch 06/8 | train_loss 1.1773 acc 0.6040 | val_loss 1.3121 acc 0.5804 | 31.7s
  epoch 07/8 | train_loss 1.1516 acc 0.6228 | val_loss 1.2943 acc 0.5804 | 32.6s
  epoch 08/8 | train_loss 1.1198 acc 0.6243 | val_loss 1.2685 acc 0.5893 | 31.7s
  >>> EfficientNet-B0 done in 4.6 min


,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,AlexNet,37.29,25.36,37.29,28.81,82.21
1,VGG16,36.44,44.71,36.44,36.05,78.60
2,VGG19,34.75,25.72,34.75,27.17,77.92
3,ResNet18,40.68,49.21,40.68,34.63,82.25
4,ResNet50,33.90,43.65,33.90,27.86,82.16
5,ResNet101,39.83,48.94,39.83,33.60,81.36
6,DenseNet121,44.07,56.72,44.07,41.56,83.90
7,EfficientNet-B0,40.68,43.94,40.68,36.92,80.63


In [9]:
# ============================================================
# 9. Deep feature extraction for Table 2
# ============================================================
# Use the backbone with the best Table-1 accuracy as the fixed feature extractor.
best_model_name = table1.sort_values("Accuracy (%)", ascending=False).iloc[0]["Model"]
print("Best backbone for feature extraction:", best_model_name)

feat_model, feat_dim = build_model(best_model_name)
feat_model.load_state_dict(trained_models[best_model_name].state_dict())
feat_model = feat_model.to(device)
feat_model.eval()

# Strip the final classification layer so forward() returns the penultimate features
def strip_head(model, name):
    name = name.lower()
    if name in ("alexnet", "vgg16", "vgg19"):
        model.classifier = model.classifier[:-1]
    elif name in ("resnet18", "resnet50", "resnet101"):
        model.fc = nn.Identity()
    elif name == "densenet121":
        model.classifier = nn.Identity()
    elif name == "efficientnet-b0":
        model.classifier = model.classifier[:-1]
    return model

feat_model = strip_head(feat_model, best_model_name)

@torch.no_grad()
def extract_features(model, loader):
    feats, labels = [], []
    for x, y in loader:
        x = x.to(device)
        f = model(x)
        f = torch.flatten(f, 1)
        feats.append(f.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# Use eval-transform train set (no augmentation) for stable features
train_feat_loader = DataLoader(Subset(full_train_eval_ds, train_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

X_train, y_train = extract_features(feat_model, train_feat_loader)
X_test,  y_test  = extract_features(feat_model, test_loader)
print("Feature shapes:", X_train.shape, X_test.shape)


Best backbone for feature extraction: DenseNet121
Feature shapes: (2015, 1024) (118, 1024)


In [10]:
# ============================================================
# 10. TABLE 2 — classical classifiers on deep features
# ============================================================
# SVC (both kernels) scales roughly O(n^2)-O(n^3) with sample count and is
# by far the slowest part of this cell on deep-feature vectors with 1000s of
# rows. We subsample the *training* features for the two SVMs only; every
# other classifier still trains on the full feature set. Raise SVM_MAX_N (or
# just drop the subsampling) if you have time to spare and want the exact
# full-data result.
SVM_MAX_N = 4000
if len(X_train) > SVM_MAX_N:
    rng = np.random.RandomState(SEED)
    svm_idx = rng.choice(len(X_train), size=SVM_MAX_N, replace=False)
    X_train_svm, y_train_svm = X_train[svm_idx], y_train[svm_idx]
else:
    X_train_svm, y_train_svm = X_train, y_train

classifiers_full_data = {
    "Logistic Regression": LogisticRegression(max_iter=2000, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, tree_method="hist",
                              eval_metric="mlogloss", random_state=SEED, n_jobs=-1),
}

# LinearSVC has no predict_proba, so wrap it in a probability calibrator.
# Both are trained on the (possibly subsampled) SVM training set.
classifiers_svm_data = {
    "Linear SVM": CalibratedClassifierCV(LinearSVC(max_iter=5000, random_state=SEED), cv=3),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
}

table2_rows = []
for clf_name, clf in classifiers_full_data.items():
    t0 = time.time()
    print(f"Fitting {clf_name} on {len(X_train)} samples...")
    clf.fit(X_train, y_train)
    probs = clf.predict_proba(X_test)
    metrics = compute_metrics(y_test, probs)
    metrics["Feature Extractor"] = "Deep Features"
    metrics["Classifier"] = clf_name
    table2_rows.append(metrics)
    print(f"  done in {time.time()-t0:.1f}s")

for clf_name, clf in classifiers_svm_data.items():
    t0 = time.time()
    print(f"Fitting {clf_name} on {len(X_train_svm)} samples (subsampled)...")
    clf.fit(X_train_svm, y_train_svm)
    probs = clf.predict_proba(X_test)
    metrics = compute_metrics(y_test, probs)
    metrics["Feature Extractor"] = "Deep Features"
    metrics["Classifier"] = clf_name
    table2_rows.append(metrics)
    print(f"  done in {time.time()-t0:.1f}s")

table2 = pd.DataFrame(table2_rows)[["Feature Extractor", "Classifier", "Accuracy (%)",
                                     "Precision (%)", "Recall (%)", "F1-Score (%)", "AUC (%)"]]
table2 = table2.round(2)
table2


Fitting Logistic Regression on 2015 samples...


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  done in 8.8s
Fitting Decision Tree on 2015 samples...
  done in 2.0s
Fitting Random Forest on 2015 samples...
  done in 8.2s
Fitting K-Nearest Neighbors (KNN) on 2015 samples...
  done in 0.1s
Fitting XGBoost on 2015 samples...
  done in 134.8s
Fitting Linear SVM on 2015 samples (subsampled)...
  done in 30.1s
Fitting RBF-SVM on 2015 samples (subsampled)...


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  done in 12.3s


,Feature Extractor,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,Deep Features,Logistic Regression,47.46,54.54,47.46,45.18,84.12
1,Deep Features,Decision Tree,25.42,25.46,25.42,23.78,56.94
2,Deep Features,Random Forest,35.59,43.43,35.59,25.09,80.35
3,Deep Features,K-Nearest Neighbors (KNN),39.83,42.94,39.83,38.34,75.52
4,Deep Features,XGBoost,42.37,42.97,42.37,37.46,78.98
5,Deep Features,Linear SVM,45.76,39.38,45.76,38.61,85.18
6,Deep Features,RBF-SVM,49.15,53.84,49.15,45.20,87.08


In [11]:
# ============================================================
# 11. TABLE 3 — computational efficiency
# ============================================================
def model_size_mb(model):
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024 ** 2)

def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def measure_inference_ms(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), n_warmup=10, n_runs=50):
    model = model.to(device).eval()
    dummy = torch.randn(input_size).to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_runs):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.time()
    return (end - start) / n_runs * 1000  # ms per image

table3_rows = []
acc_lookup = table1.set_index("Model")["Accuracy (%)"].to_dict()

for name in MODEL_NAMES_T3:
    print(f"Profiling {name} ...")
    model, _ = build_model(name)
    model.load_state_dict(trained_models[name].state_dict())
    model = model.to(device)

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    flops, _ = profile(copy.deepcopy(model), inputs=(dummy,), verbose=False)
    gflops = flops / 1e9

    row = {
        "Model": name,
        "Parameters (M)": round(count_params_m(model), 2),
        "Model Size (MB)": round(model_size_mb(model), 2),
        "FLOPs (G)": round(gflops, 2),
        "Inference Time (ms)": round(measure_inference_ms(model), 2),
        "Accuracy (%)": round(acc_lookup[name], 2),
    }
    table3_rows.append(row)
    model.cpu()
    torch.cuda.empty_cache()

table3 = pd.DataFrame(table3_rows)[["Model", "Parameters (M)", "Model Size (MB)",
                                     "FLOPs (G)", "Inference Time (ms)", "Accuracy (%)"]]
table3


Profiling AlexNet ...
Profiling VGG16 ...
Profiling VGG19 ...
Profiling ResNet18 ...
Profiling ResNet50 ...
Profiling DenseNet121 ...
Profiling EfficientNet-B0 ...


,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,AlexNet,57.04,217.59,0.71,2.05,37.29
1,VGG16,134.30,512.30,15.47,9.60,36.44
2,VGG19,139.61,532.56,19.63,11.42,34.75
3,ResNet18,11.18,42.69,1.82,2.06,40.68
4,ResNet50,23.53,89.95,4.13,7.52,33.90
5,DenseNet121,6.96,26.88,2.90,20.66,44.07
6,EfficientNet-B0,4.02,15.49,0.41,7.65,40.68


In [12]:
# ============================================================
# 12. Export results (matches the tables in Task_01.docx)
# ============================================================
with pd.ExcelWriter("results_tables.xlsx") as writer:
    table1.to_excel(writer, sheet_name="Table1_TransferLearning", index=False)
    table2.to_excel(writer, sheet_name="Table2_Classifiers", index=False)
    table3.to_excel(writer, sheet_name="Table3_Efficiency", index=False)

table1.to_csv("table1_transfer_learning.csv", index=False)
table2.to_csv("table2_classifiers.csv", index=False)
table3.to_csv("table3_efficiency.csv", index=False)

print("Saved: results_tables.xlsx, table1_transfer_learning.csv, table2_classifiers.csv, table3_efficiency.csv")

print("\n--- Table 1: Transfer Learning Models ---")
print(table1.to_markdown(index=False))
print("\n--- Table 2: Different Classifiers ---")
print(table2.to_markdown(index=False))
print("\n--- Table 3: Computational Efficiency ---")
print(table3.to_markdown(index=False))


Saved: results_tables.xlsx, table1_transfer_learning.csv, table2_classifiers.csv, table3_efficiency.csv

--- Table 1: Transfer Learning Models ---
| Model           |   Accuracy (%) |   Precision (%) |   Recall (%) |   F1-Score (%) |   AUC (%) |
|:----------------|---------------:|----------------:|-------------:|---------------:|----------:|
| AlexNet         |          37.29 |           25.36 |        37.29 |          28.81 |     82.21 |
| VGG16           |          36.44 |           44.71 |        36.44 |          36.05 |     78.6  |
| VGG19           |          34.75 |           25.72 |        34.75 |          27.17 |     77.92 |
| ResNet18        |          40.68 |           49.21 |        40.68 |          34.63 |     82.25 |
| ResNet50        |          33.9  |           43.65 |        33.9  |          27.86 |     82.16 |
| ResNet101       |          39.83 |           48.94 |        39.83 |          33.6  |     81.36 |
| DenseNet121     |          44.07 |           56.72 |       

**Debug after low accuracy?**
1: Class Mismatch
2: Frozen ImageNet features genuinely aren't good enough for this task.

In [14]:
print("Train classes:", full_train_ds.classes)
print("Test classes: ", test_ds.classes)
print("Match:", full_train_ds.classes == test_ds.classes)
print("Train class_to_idx:", full_train_ds.class_to_idx)
print("Test class_to_idx: ", test_ds.class_to_idx)

Train classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Test classes:  ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Match: True
Train class_to_idx: {'actinic keratosis': 0, 'basal cell carcinoma': 1, 'dermatofibroma': 2, 'melanoma': 3, 'nevus': 4, 'pigmented benign keratosis': 5, 'seborrheic keratosis': 6, 'squamous cell carcinoma': 7, 'vascular lesion': 8}
Test class_to_idx:  {'actinic keratosis': 0, 'basal cell carcinoma': 1, 'dermatofibroma': 2, 'melanoma': 3, 'nevus': 4, 'pigmented benign keratosis': 5, 'seborrheic keratosis': 6, 'squamous cell carcinoma': 7, 'vascular lesion': 8}


**Classes do not mismatch, possibility 2 is true**

Cannot proceed as runtime will go up — each model now does a backward pass through every layer, not just the head. Budget roughly 10-25 min per model on a T4, so 1.5-3 hours total for all 8, versus the ~35 min from the frozen version. Watch the per-epoch printouts (loss/acc trend) — if a model's val_loss is still dropping right at epoch 20, that one might benefit from more epochs.